# Data Catalog / Knowledge Catalog — Policy Tags & Column-Level Security Demo

Companion notebook to the **Data Catalog, Policy Tags & Column-Level Security Handbook**.

This notebook builds a small BigQuery `employees` table, classifies two sensitive columns (`email`, `salary`) with policy tags, turns on column-level access control, and then proves enforcement by querying as a restricted principal.

**Naming note (July 2026):** the standalone *Data Catalog* product was deprecated Jan 30, 2026 and its functionality now lives inside *Knowledge Catalog* (formerly Dataplex Universal Catalog, renamed April 2026). The Python client library package name (`google-cloud-datacatalog`), the API, and all resource paths used below are unchanged — only the console branding moved.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet google-cloud-datacatalog google-cloud-bigquery google-cloud-bigquery-datapolicies


In [ ]:
# If running in Colab, authenticate as yourself (must have Owner / BigQuery Admin +
# Data Catalog Policy Tag Admin + Data Catalog Admin on the project).
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit these before running


In [ ]:
PROJECT_ID = "himanshugcpproject"   # <-- your GCP project
LOCATION = "us"                     # taxonomy region; must match the BigQuery dataset region
DATASET_ID = "hr_demo"
TABLE_ID = "employees"
TAXONOMY_DISPLAY_NAME = "HR Data Classification"

# A second Google account or group you control, used to demonstrate restricted access.
# Format: "user:[email protected]" or "group:[email protected]"
TEST_PRINCIPAL = "user:[email protected]"

print(f"Project: {PROJECT_ID} | Location: {LOCATION} | Dataset.Table: {DATASET_ID}.{TABLE_ID}")


## 2. Create the dataset and sample table


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
dataset = bq_client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {dataset.dataset_id} ({dataset.location})")


In [ ]:
schema = [
    bigquery.SchemaField("employee_id", "INT64"),
    bigquery.SchemaField("full_name", "STRING"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("salary", "NUMERIC"),
]

table_ref = bigquery.TableReference(dataset_ref, TABLE_ID)
table = bigquery.Table(table_ref, schema=schema)
table = bq_client.create_table(table, exists_ok=True)
print(f"Table ready: {table.table_id}")

rows = [
    {"employee_id": 1, "full_name": "Riya Sharma", "department": "Engineering", "email": "[email protected]", "salary": 950000},
    {"employee_id": 2, "full_name": "Arjun Verma", "department": "Sales", "email": "[email protected]", "salary": 720000},
    {"employee_id": 3, "full_name": "Meera Nair", "department": "Finance", "email": "[email protected]", "salary": 880000},
]
errors = bq_client.insert_rows_json(table, rows)
print("Insert errors:", errors if errors else "none")


## 3. Create the taxonomy and policy tags


In [ ]:
from google.cloud import datacatalog_v1

dc_client = datacatalog_v1.PolicyTagManagerClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

taxonomy = datacatalog_v1.Taxonomy()
taxonomy.display_name = TAXONOMY_DISPLAY_NAME
taxonomy.description = "Classification for sensitive HR columns (demo)"
taxonomy.activated_policy_types = [
    datacatalog_v1.Taxonomy.PolicyType.FINE_GRAINED_ACCESS_CONTROL
]

created_taxonomy = dc_client.create_taxonomy(parent=parent, taxonomy=taxonomy)
print("Taxonomy created:", created_taxonomy.name)


In [ ]:
def create_policy_tag(display_name, description, parent_policy_tag=None):
    tag = datacatalog_v1.PolicyTag()
    tag.display_name = display_name
    tag.description = description
    if parent_policy_tag:
        tag.parent_policy_tag = parent_policy_tag
    return dc_client.create_policy_tag(parent=created_taxonomy.name, policy_tag=tag)

pii_tag = create_policy_tag("PII", "Personally identifiable information")
financial_tag = create_policy_tag("Financial", "Compensation and financial data")

print("PII policy tag:", pii_tag.name)
print("Financial policy tag:", financial_tag.name)


## 4. Attach the policy tags to table columns

BigQuery's schema-update API is used to attach a `policy_tags` entry to a specific column.


In [ ]:
new_schema = []
for field in table.schema:
    if field.name == "email":
        new_schema.append(bigquery.SchemaField(
            field.name, field.field_type,
            policy_tags=bigquery.PolicyTagList(names=[pii_tag.name])
        ))
    elif field.name == "salary":
        new_schema.append(bigquery.SchemaField(
            field.name, field.field_type,
            policy_tags=bigquery.PolicyTagList(names=[financial_tag.name])
        ))
    else:
        new_schema.append(field)

table.schema = new_schema
table = bq_client.update_table(table, ["schema"])
print("Schema updated. Policy tags attached to: email -> PII, salary -> Financial")
for f in table.schema:
    tag_info = f.policy_tags.names if f.policy_tags else None
    print(f"  {f.name}: {tag_info}")


## 5. Enforce access control on the taxonomy

Until this step, the policy tags are descriptive labels only — nothing is blocked yet.

The Python client library does not expose a single convenience call for toggling enforcement, so this cell calls the REST API directly (same effect as the console's 'Enforce access control' toggle).


In [ ]:
import google.auth
import google.auth.transport.requests
import requests

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())

taxonomy_id = created_taxonomy.name.split("/")[-1]
url = (
    f"https://datacatalog.googleapis.com/v1/{created_taxonomy.name}"
    f"?updateMask=activatedPolicyTypes"
)
headers = {
    "Authorization": f"Bearer {creds.token}",
    "Content-Type": "application/json",
}
payload = {"activatedPolicyTypes": ["FINE_GRAINED_ACCESS_CONTROL"]}

resp = requests.patch(url, headers=headers, json=payload)
print(resp.status_code, resp.json())


## 6. Grant Fine-Grained Reader on the PII tag to the test principal

This intentionally grants access to **PII only**, not **Financial**, so the demo can show partial access: the test principal will be able to read `email` but denied on `salary`.


In [ ]:
from google.iam.v1 import policy_pb2

policy = dc_client.get_iam_policy(request={"resource": pii_tag.name})
binding = policy_pb2.Binding(
    role="roles/datacatalog.categoryFineGrainedReader",
    members=[TEST_PRINCIPAL],
)
policy.bindings.append(binding)

updated_policy = dc_client.set_iam_policy(
    request={"resource": pii_tag.name, "policy": policy}
)
print(f"Granted roles/datacatalog.categoryFineGrainedReader on PII tag to {TEST_PRINCIPAL}")
print(updated_policy)


**Also make sure** `TEST_PRINCIPAL` has ordinary BigQuery access to the dataset (policy tags are an *additional* layer, not a replacement for normal ACLs):


In [ ]:
entry = bigquery.AccessEntry(
    role="READER",
    entity_type="userByEmail" if TEST_PRINCIPAL.startswith("user:") else "groupByEmail",
    entity_id=TEST_PRINCIPAL.split(":", 1)[1],
)
dataset = bq_client.get_dataset(dataset_ref)
entries = list(dataset.access_entries)
entries.append(entry)
dataset.access_entries = entries
dataset = bq_client.update_dataset(dataset, ["access_entries"])
print(f"Granted dataset-level READER on {DATASET_ID} to {TEST_PRINCIPAL}")


## 7. Test enforcement

Run these two queries **as your own admin account** first (they should both succeed if you've also granted yourself Fine-Grained Reader on both tags), then re-run signed in as `TEST_PRINCIPAL` (e.g. in a second Colab session, or via `bq query` after `gcloud auth login` as that account) to see the second query fail.


In [ ]:
query_allowed = f"""
SELECT employee_id, full_name, email
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""
print("Query touching only the PII-tagged column (email):")
try:
    for row in bq_client.query(query_allowed).result():
        print(dict(row))
except Exception as e:
    print("FAILED:", e)


In [ ]:
query_restricted = f"""
SELECT employee_id, salary
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""
print("Query touching the Financial-tagged column (salary):")
try:
    for row in bq_client.query(query_restricted).result():
        print(dict(row))
except Exception as e:
    print("FAILED (expected if this account lacks Fine-Grained Reader on Financial):", e)


## 8. Cleanup

Run this cell to remove everything the notebook created, so the demo project stays tidy.


In [ ]:
CONFIRM_CLEANUP = False  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    dc_client.delete_taxonomy(name=created_taxonomy.name)
    print("Taxonomy (and its policy tags) deleted.")
    bq_client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
    print("Dataset deleted.")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
